#### Basic

### Steps
    <!-- count adjacent pairs
    find the most frequent pair
    merge it everywhere
    repeat -->

In [5]:
## Example of a simple corpus for testing purposes
corpus = [
    "low lower lowest",
    "newer wider",
    "low low lower"
]

In [6]:
## Convert text into a word-frequency vocabulary

from collections import Counter

# Create a vocabulary of symbols from the corpus
vocab = Counter()

# Iterate through each sentence in the corpus
# Output the vocabulary of symbols and their frequencies
for sentence in corpus:
    for word in sentence.split():
        symbols = tuple(list(word) + ["</w>"])
        vocab[symbols] += 1

vocab

Counter({('l', 'o', 'w', '</w>'): 3,
         ('l', 'o', 'w', 'e', 'r', '</w>'): 2,
         ('l', 'o', 'w', 'e', 's', 't', '</w>'): 1,
         ('n', 'e', 'w', 'e', 'r', '</w>'): 1,
         ('w', 'i', 'd', 'e', 'r', '</w>'): 1})

In [7]:
## Count Every adjecent Pair of Symbols in the Vocabulary 

def get_pair_counts(vocab):
    pair_counts = Counter()

    for word, frequency in vocab.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += frequency

    return pair_counts

pair_counts = get_pair_counts(vocab)
pair_counts   

Counter({('l', 'o'): 6,
         ('o', 'w'): 6,
         ('w', 'e'): 4,
         ('e', 'r'): 4,
         ('r', '</w>'): 4,
         ('w', '</w>'): 3,
         ('e', 's'): 1,
         ('s', 't'): 1,
         ('t', '</w>'): 1,
         ('n', 'e'): 1,
         ('e', 'w'): 1,
         ('w', 'i'): 1,
         ('i', 'd'): 1,
         ('d', 'e'): 1})

In [8]:
## Merge the most frequent pair

def merge_pair(pair, vocab):
    new_vocab = Counter()

    for word, frequency in vocab.items():
        merged_word = []
        i = 0

        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                merged_word.append(word[i] + word[i + 1])
                i += 2
            else:
                merged_word.append(word[i])
                i += 1

        new_vocab[tuple(merged_word)] += frequency

    return new_vocab

In [9]:
## Train BPE by repeatedly finding and merging the best pair
num_merges = 10
merges = []

for _ in range(num_merges):
    pair_counts = get_pair_counts(vocab)

    if not pair_counts:
        break

    best_pair = max(pair_counts, key=pair_counts.get)

    print("Merging:", best_pair, "count:", pair_counts[best_pair])

    vocab = merge_pair(best_pair, vocab)
    merges.append(best_pair)

Merging: ('l', 'o') count: 6
Merging: ('lo', 'w') count: 6
Merging: ('e', 'r') count: 4
Merging: ('er', '</w>') count: 4
Merging: ('low', '</w>') count: 3
Merging: ('low', 'er</w>') count: 2
Merging: ('low', 'e') count: 1
Merging: ('lowe', 's') count: 1
Merging: ('lowes', 't') count: 1
Merging: ('lowest', '</w>') count: 1


In [10]:
## Encode new text using the learned merges

def encode_word(word, merges):
    tokens = list(word) + ["</w>"]

    for first, second in merges:
        new_tokens = []
        i = 0

        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == first and tokens[i + 1] == second:
                new_tokens.append(first + second)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens

encode_word("lower", merges)

['lower</w>']

In [11]:
## Encode a whole sentence

def encode_text(text, merges):
    all_tokens = []

    for word in text.split():
        all_tokens.extend(encode_word(word, merges))

    return all_tokens

tokens = encode_text("low lower", merges)
tokens

['low</w>', 'lower</w>']

In [12]:
# Decode the tokens back into a string

def decode(tokens):
    return "".join(tokens).replace("</w>", " ").strip()

decode(tokens)

'low lower'

#### Method 2:

In [13]:
from collections import Counter
import json

In [14]:
## Example of a simple corpus for testing purposes
corpus = [
    "low lower lowest",
    "newer wider",
    "low low lower"
]

In [15]:
END_OF_WORD = "</w>"
UNK_TOKEN = "<unk>"

In [16]:
def build_vocab(corpus):
    """Convert corpus text into character-based word vocabulary."""
    vocab = Counter()

    for sentence in corpus:
        for word in sentence.split():
            tokens = tuple(list(word) + [END_OF_WORD])
            vocab[tokens] += 1

    return vocab

In [17]:
def get_pair_counts(vocab):
    """Count every adjacent token pair, weighted by word frequency."""
    pair_counts = Counter()

    for word_tokens, frequency in vocab.items():
        for i in range(len(word_tokens) - 1):
            pair = (word_tokens[i], word_tokens[i + 1])
            pair_counts[pair] += frequency

    return pair_counts

In [18]:
def merge_pair(pair, vocab):
    """Merge one selected pair throughout the vocabulary."""
    new_vocab = Counter()

    for word_tokens, frequency in vocab.items():
        merged_tokens = []
        i = 0

        while i < len(word_tokens):
            is_pair = (
                i < len(word_tokens) - 1
                and word_tokens[i] == pair[0]
                and word_tokens[i + 1] == pair[1]
            )

            if is_pair:
                merged_tokens.append(pair[0] + pair[1])
                i += 2
            else:
                merged_tokens.append(word_tokens[i])
                i += 1

        new_vocab[tuple(merged_tokens)] += frequency

    return new_vocab

In [19]:
def train_bpe(corpus, num_merges=10):
    """Learn BPE merge rules and return merges plus the token vocabulary."""
    vocab = build_vocab(corpus)

    # Keep original characters so known characters never become <unk>
    token_vocab = {UNK_TOKEN, END_OF_WORD}

    for word_tokens in vocab:
        token_vocab.update(word_tokens)

    merges = []

    for _ in range(num_merges):
        pair_counts = get_pair_counts(vocab)

        if not pair_counts:
            break

        # Highest frequency; lexical order resolves ties consistently.
        best_pair = min(
            pair_counts,
            key=lambda pair: (-pair_counts[pair], pair)
        )

        vocab = merge_pair(best_pair, vocab)
        merges.append(best_pair)

    # Add every learned merged token.
    for word_tokens in vocab:
        token_vocab.update(word_tokens)

    return merges, token_vocab

In [20]:
def encode_word(word, merges, token_vocab):
    """Convert one word into BPE tokens."""
    tokens = list(word) + [END_OF_WORD]

    # Replay merge rules in exactly the order used during training.
    for first, second in merges:
        new_tokens = []
        i = 0

        while i < len(tokens):
            is_pair = (
                i < len(tokens) - 1
                and tokens[i] == first
                and tokens[i + 1] == second
            )

            if is_pair:
                new_tokens.append(first + second)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    # Any unseen character/token becomes <unk>.
    return [
        token if token in token_vocab else UNK_TOKEN
        for token in tokens
    ]

In [21]:
def encode_text(text, merges, token_vocab):
    """Convert a sentence into BPE tokens."""
    all_tokens = []

    for word in text.split():
        all_tokens.extend(encode_word(word, merges, token_vocab))

    return all_tokens

In [22]:
def decode(tokens):
    """Turn BPE tokens back into readable text."""
    return "".join(tokens).replace(END_OF_WORD, " ").strip()

In [23]:
def save_model(filename, merges, token_vocab, num_merges):
    """Save the trained tokenizer model."""
    model = {
        "num_merges": num_merges,
        "merges": merges,
        "token_vocab": sorted(token_vocab)
    }

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(model, file, ensure_ascii=False, indent=2)


In [24]:
def load_model(filename):
    """Load a previously saved tokenizer model."""
    with open(filename, "r", encoding="utf-8") as file:
        model = json.load(file)

    merges = [tuple(pair) for pair in model["merges"]]
    token_vocab = set(model["token_vocab"])

    return merges, token_vocab, model["num_merges"]

In [25]:
corpus = [
    "low lower lowest",
    "newer wider",
    "low low lower"
]

num_merges = 10

merges, token_vocab = train_bpe(corpus, num_merges)

tokens = encode_text("low lower zoo", merges, token_vocab)

print("Merges:", merges)
print("Tokens:", tokens)
print("Decoded:", decode(tokens))

Merges: [('l', 'o'), ('lo', 'w'), ('e', 'r'), ('er', '</w>'), ('low', '</w>'), ('low', 'er</w>'), ('d', 'er</w>'), ('e', 's'), ('e', 'w'), ('es', 't')]
Tokens: ['low</w>', 'lower</w>', '<unk>', 'o', 'o', '</w>']
Decoded: low lower <unk>oo


In [26]:
save_model(
    "bpe_model.json",
    merges,
    token_vocab,
    num_merges
)

loaded_merges, loaded_vocab, loaded_num_merges = load_model("bpe_model.json")

tokens = encode_text(
    "lowest zoo",
    loaded_merges,
    loaded_vocab
)

print(tokens)
print(decode(tokens))

['low', 'est', '</w>', '<unk>', 'o', 'o', '</w>']
lowest <unk>oo
